# Agent-Based Model: Student Housing & Loneliness

This notebook simulates how students choose between studio and shared housing, and how that choice, shaped by rents, supply, and policy decisions, feeds back into loneliness, affordability, and the public budget.

The **Control Panel** section contains the main settings for the model runs, and the key parameters.

1. Research question & policy aim
2. Conceptual model overview
3. Key assumptions
4. Control Panel (all the knobs) & parameter table
5. Model implementation
6. Baseline run & sanity checks
7. Policy scenarios
8. Multi-seed evaluation
9. Marginal contribution of each lever (ablation)
10. Local sensitivity analysis
11. Results plots - policy set comparisons
12. Robustness of the scenario ranking (ranking plots)
13. Global sensitivity analysis
14. Interpretation and limitations
16. Possible extensions
17. Conclusion


## 1. Modelling questiosn & policy aim

- How do students choose between studios and shared housing?
- How does that choice affect loneliness?
- How do different **policy scenarios** – combinations for different variables – impact key outcomes (defined later)

Our aim is not prediction since we don't have enough data. The hope is that we can simulate some changes in scenarios and see the trade-offs between them, and the lifecycle dyanmics.

## 2. Conceptual overview of model

Each **student** has an income and a set of fixed traits (their privacy preferences, their social need, sensitivities to price, loneliness, and financial stress). Each month/time period in the model they choose the best feasible option. This in turn updates their loneliness based on mismatch, isolation, financial stress, and day-to-day interaction.

The **market** sets rents endogenously (rents rise with scarcity, fall with vacancy), lets landlords convert or withdraw units when renting out is unprofitable, and lets developers commission new units that become available with a time lag.

**Policy** is part of the model through a few levers reported on: studio rent allowance (huurtoeslag), a potential reduction to that allowance, a similar rent allowance for rooms in shared units, an developed-directed 'object subsidy' for building shared rooms, and a cross-finance mechanism that funds construction from government's savings on rental allowance.

## 3. Model assumptions

- One step = one month. Students re-enter the market only when unhoused or mismatched and willing to move.
- Housing choice is a forced choice: if any affordable option is available, the student takes the highest-utility one rather than waiting. Having any form of housing is preferable to none.
- Underlying preference is fixed per student (they structurally prefer one form over another depending on personality), but the *revealed* preference shifts with the effective cost gap between studio and shared housing. ie. they can trade off different kinds of costs against each other
- Rents are sticky / have inertia, and somewhat bounded (these can be set extremely low or high to test)
- Loneliness decays over time and fluctuates around a personal baseline (Cacioppo & Hawkley, 2010).

## Environment set-up

In [ ]:
print(
    "Using packages defined in requirements.txt. Virutal environment is recommended."
)

choice = input(
    "Install/update packages from requirements.txt? [y/N]: "
).strip().lower()

if choice != "y":
    raise SystemExit("Stopped. Nothing was installed.")

%pip install -r requirements.txt

print("\nIf anything got updated, restart the kernel before continuing.")

### Imports

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mesa

from mesa.datacollection import DataCollector

## 4. Control Panel - global variables

This cell provides an overview of all settings needed for the model. Each of the capitalized dictionaries control a specific domain:
- `RUN_CONFIG` controls how the simulation is run
- `MODEL_PARAMS` holds the baseline settings that every scenario starts from (which adding or changing specific settings dictated by policy decisions)
- `INCOME_DECILE...` are income values based on eurostudent.eu data, from which we backcalculate a realistic income distribution.

In [ ]:
# 1. Simulation run settings
RUN_CONFIG = {
    "steps": 300,          # months simulated per run (1 step = 1 month)
    "seeds": range(10),    # how many random seeds per scenario (more = smoother, slower)
    "window": 50,          # outcomes are averaged over the final `window` steps
}

# 2. Student income distribution (empirical input)
# Source: database.eurostudent.eu. Per-decile upper bound and mean monthly income.
INCOME_DECILE_MAX_VALUES = [
    780, 1000, 1150, 1301.67, 1461.67,
    1618.33, 1810, 2055, 2425, 6556.67,
]
INCOME_DECILE_MEAN_VALUES = [
    567.24, 893.87, 1074.81, 1228.65, 1384.45,
    1537.44, 1716.70, 1930.14, 2229.87, 2983.56,
]

# 3. Baseline model parameters - passed to every run but can be overridden by scenario-specific settings
MODEL_PARAMS = {
    # Population & initial housing stock
    "n_students": 300,
    "initial_studios": 75,
    "initial_shared": 129,
    "shared_household_size": 4,        # people per shared unit

    # Student behaviour & wellbeing
    "move_probability": 0.25,          # chance a mismatched, housed student tries to move
    "reference_gap": 200.0,            # € euro cost gap that can shift a student's preference between studio/shared according to LMS data
    "interaction_quality_studio": 0.31,
    "interaction_quality_shared": 0.72,

    # Housing market dynamics (rents)
    "rent_adjustment_speed": 0.017,    # how fast rents rise with scarcity
    "vacancy_sensitivity": 0.03,       # how fast rents fall with vacancy
    "rent_inertia": 0.2,               # smoothing on rent updates (0 = jumpy, 1 = instant)
    "studio_rent_eq": 750.0,           # starting / reference studio rent (€)
    "shared_rent_eq": 500.0,           # starting / reference shared rent (€)
    "min_studio_rent": 450.0, "max_studio_rent": 1500.0,
    "min_shared_rent": 420.0, "max_shared_rent": 1000.0,
    "natural_vacancy_studio": 0.03,
    "natural_vacancy_shared": 0.03,

    # Landlord behaviour
    "operating_cost": 500.0,           # monthly cost per unit (€)
    "coverage_ratio": 0.90,            # income/operating-cost ratio below which landlords exit
    "conversion_rate": 0.10,           # fraction of stock converted between types per step
    "investor_withdrawal_rate": 0.05,  # fraction withdrawn when unprofitable

    # Developer / construction
    "studio_build_cost": 486.0,        # amortised monthly build cost (€)
    "shared_build_cost": 347.0,
    "gamma_studio": 0.03, "gamma_shared": 0.03,   # build responsiveness
    "kappa_buffer_pct": 0.20,          # profit buffer required before building
    "construction_lag_ticks": 24,      # months between commissioning and delivery

    # Outcome-measure assumptions
    "studio_m2_per_person": 25.0,
    "shared_m2_per_person": 12.0,
    "low_income_threshold": 1000.0,    # €/month defining "low income" for access metric
    "privacy_lonely_threshold": 0.7,   # privacy_pref above which a studio resident is "at risk"

    # Policy levers (off by default)
    "studio_allowance_regime": "none",  # "none" | "pre_2026" | "post_2026"
    "studio_allowance_reduction": 0.0,
    "shared_rent_allowance": 0.0,
    "object_subsidy_per_room": 0.0,
    "budget_cap_per_year": float("inf"),
    "crossfinance_enabled": False,
}

### Parameter documentation

| Group | Knob | Meaning | Default | Source / note |
|---|---|---|---|---|
| Run | `steps` | Months per run | 300 | 1 step = 1 month |
| Run | `seeds` | Random seeds per scenario | `range(10)` | more = smoother, slower |
| Run | `window` | Final steps averaged for outcomes | 50 | end-state summary |
| Population | `n_students` | Number of student agents | 300 | |
| Stock | `initial_studios` / `initial_shared` | Starting units | 75 / 129 | |
| Stock | `shared_household_size` | People per shared unit | 4 | |
| Behaviour | `move_probability` | Chance a mismatched student moves | 0.25 | |
| Behaviour | `reference_gap` | €gap giving full preference pull | 200 | LMS 2025; Kences et al. |
| Behaviour | `interaction_quality_studio` / `_shared` | Interaction quality by type | 0.31 / 0.72 | |
| Market | `rent_adjustment_speed` | Rent rise per unit scarcity | 0.017 | |
| Market | `vacancy_sensitivity` | Rent fall per unit vacancy | 0.03 | |
| Market | `rent_inertia` | Rent-update smoothing | 0.2 | |
| Market | `studio_rent_eq` / `shared_rent_eq` | Starting/reference rents (€) | 750 / 500 | |
| Market | rent floors/ceilings | Rent bounds (€) | 450–1500 / 420–1000 | |
| Landlord | `operating_cost` | Monthly cost per unit (€) | 500 | |
| Landlord | `coverage_ratio` | Exit threshold (income/cost) | 0.90 | |
| Landlord | `conversion_rate` | Stock converted per step | 0.10 | |
| Landlord | `investor_withdrawal_rate` | Stock withdrawn if unprofitable | 0.05 | |
| Developer | `studio_build_cost` / `shared_build_cost` | Amortised monthly build cost (€) | 486 / 347 | |
| Developer | `gamma_studio` / `gamma_shared` | Build responsiveness | 0.03 | |
| Developer | `kappa_buffer_pct` | Profit buffer before building | 0.20 | |
| Developer | `construction_lag_ticks` | Months to delivery | 24 | |
| Outcomes | `studio_m2_per_person` / `shared_m2_per_person` | Space per person (m²) | 25 / 12 | |
| Outcomes | `low_income_threshold` | €defining low income | 1000 | |
| Outcomes | `privacy_lonely_threshold` | Privacy pref flagging at-risk studio resident | 0.7 | |
| Policy | `studio_allowance_regime` | huurtoeslag regime | `none` | `none`/`pre_2026`/`post_2026` |
| Policy | `studio_allowance_reduction` | €cut from studio allowance | 0 | scenario-driven |
| Policy | `shared_rent_allowance` | €allowance for shared rooms | 0 | scenario-driven |
| Policy | `object_subsidy_per_room` | €subsidy per built room | 0 | scenario-driven |
| Policy | `budget_cap_per_year` | Annual subsidy budget (€) | ∞ | scenario-driven |
| Policy | `crossfinance_enabled` | Fund builds from allowance savings | False | scenario-driven |

Some utility weights that are not globally-relevant have been added within the agent functions, marked by comments.

## 5. Model implementation

Defining:
- the `StudentAgent`class (a simulation student), 
- the `StudentHousingModel` (the interaction between students and their environment) and 
- a `run_model` helper. 

This is a prototype of each component of the model - behaviour can be controlled from the panel above

### Generating student income distribution based on data

In [ ]:
# Draw one income per student. Each decile gets filled with a Beta-shaped spread between its bounds so the decile mean roughly holds.
def generate_student_incomes(n_agents, max_values, mean_values, min_income=0, seed=None, concentration=8):
    rng = np.random.default_rng(seed)

    max_values = np.array(max_values, dtype=float)
    mean_values = np.array(mean_values, dtype=float)

    lower_bounds = np.concatenate(([min_income], max_values[:-1]))
    upper_bounds = max_values

    incomes = []

    base_count = n_agents // 10
    remainder = n_agents % 10

    for i in range(10):
        count = base_count + (1 if i < remainder else 0)

        low = lower_bounds[i]
        high = upper_bounds[i]
        mean = mean_values[i]

        relative_mean = (mean - low) / (high - low)
        relative_mean = np.clip(relative_mean, 0.01, 0.99)

        alpha = relative_mean * concentration
        beta = (1 - relative_mean) * concentration

        samples = rng.beta(alpha, beta, size=count)
        decile_incomes = low + samples * (high - low)

        incomes.extend(decile_incomes)

    incomes = np.array(incomes)
    rng.shuffle(incomes)

    return incomes

### Trait samplers
Helper functions to generate hypothetical distributions for student traits:
- bimodal for private/open, and introverted vs extroverted
- right-skewed gaussian for time pressure

In [ ]:
def bimodal_sample(rng, mean1=0.25, std1=0.1, mean2=0.75, std2=0.1):
    # Bimodal draw, e.g. private vs. open, or introvert vs. extrovert.
    # Uses the model's seeded rng so traits come out the same for a given seed.
    return float(rng.normal(mean1, std1) if rng.random() < 0.5
                 else rng.normal(mean2, std2))


def beta_sample(rng, alpha=1.5, beta=4.0):
    # Right-skewed draw: most students low-ish, with a tail of high ones (time pressure).
    return float(rng.beta(alpha, beta))

### Student agent
Each student has an income, privacy preference, social need, time pressure, price sensitivity, and a wellbeing state. Each step they may update their preference, choose housing, and update loneliness depending on their situation.

In [ ]:
class StudentAgent(mesa.Agent):
    # One student in the housing market (no space, no neighbours).
    # Each student has a preferred housing type and, every step, has to take the
    # best option they can actually get right now. Loneliness moves with mismatch,
    # isolation, and financial stress.

    #  Anna: I didn't agree with time pressure argument, so I changed it below in the utility.

    def __init__(
        self,
        model,
        income,
        privacy_pref,
        social_need,
        time_pressure,
        loneliness_sensitivity,
        stress_sensitivity,
        price_sensitivity,
    ):
        super().__init__(model)

        self.income = income
        self.privacy_pref = privacy_pref
        self.social_need = social_need
        self.time_pressure = time_pressure
        self.loneliness_sensitivity = loneliness_sensitivity
        self.stress_sensitivity = stress_sensitivity

        self.price_sensitivity = price_sensitivity

        # latent_preference: genuine underlying preference, fixed for agent's lifetime.
        self.latent_preference = (
            "shared" if self.social_need - self.privacy_pref > 0.1 else "studio"
        )

        # housing_preference: revealed preference, starts equal to latent but
        # updates each step via update_preference() in response to subsidy structure.
        self.housing_preference = self.latent_preference
        self.current_housing = "none"
        self.mismatch = 0.0
        self.loneliness = 0.4

        self.crossfinance_pool = 0.0

        # Outcome tracking for waiting-time and match-rate measures
        self.search_time = 0
        self.total_search_time = 0
        self.ever_housed = False

        # Diagnostics (set in choose_housing; not currently collected).
        self.last_choice_utility_studio = None
        self.last_choice_utility_shared = None

    def update_preference(self):
        # Re-pick the revealed preference each step from the effective cost gap between studio and shared, scaled by how price-sensitive the student is.
        # Empirical data from (LMS 2025, Kences et al.): a ~€200/month room subsidy shifts studio preference from about 56% to 44% across all students,
        # so reference_gap (default €200) is set so a €200 gap gives a full-strength pull before the individual price sensitivity scales it.
        # cost_gap > 0 -> pull toward shared; < 0 -> pull towards studio.
        effective_studio = max(
            self.model.studio_rent - self.model.compute_studio_allowance(), 0.0
        )
        effective_shared = max(
            self.model.shared_rent - self.model.shared_rent_allowance, 0.0
        )

        reference_gap = self.model.reference_gap
        financial_pull = (effective_studio - effective_shared) / reference_gap
        # positive = pull toward shared, negative = pull toward studio

        if self.latent_preference == "studio":
            switch_prob = min(self.price_sensitivity * max(financial_pull, 0.0), 1.0)
            self.housing_preference = (
                "shared" if self.model.random.random() < switch_prob else "studio"
            )
        else:
            switch_prob = min(self.price_sensitivity * max(-financial_pull, 0.0), 1.0)
            self.housing_preference = (
                "studio" if self.model.random.random() < switch_prob else "shared"
            )

    def step(self):
        # Students don't go back on the market every step. If they're already housed they tend to stay, unless they're mismatched and willing/able to move.
        self.update_preference()

        # If already housed and matched, stay put
        if self.current_housing != "none" and self.current_housing == self.housing_preference:
            if self.current_housing == "studio":
                self.model.realized_demand_studio += 1
            elif self.current_housing == "shared":
                self.model.realized_demand_shared += 1
            return

        # If already housed but mismatched, only some try to move
        if self.current_housing != "none" and self.current_housing != self.housing_preference:
            if self.model.random.random() > self.model.move_probability:
                if self.current_housing == "studio":
                    self.model.realized_demand_studio += 1
                elif self.current_housing == "shared":
                    self.model.realized_demand_shared += 1
                return

        # If unhoused, or if mismatched and trying to move, choose housing
        old_housing = self.current_housing
        was_unhoused = old_housing == "none"
        if was_unhoused:
            self.model.seekers_this_step += 1

        choice = self.choose_housing()

        # Free up old unit if moving away from it
        if old_housing == "studio" and choice != "studio":
            self.model.available_studios += 1
        elif old_housing == "shared" and choice != "shared":
            self.model.available_shared += 1

        # Allocate new unit
        if choice == "studio" and self.model.available_studios > 0:
            self.current_housing = "studio"
            self.model.available_studios -= 1
            self.model.realized_demand_studio += 1

        elif choice == "shared" and self.model.available_shared > 0:
            self.current_housing = "shared"
            self.model.available_shared -= 1
            self.model.realized_demand_shared += 1

        else:
            self.current_housing = "none"
            self.model.realized_unhoused += 1

        # Search-time bookkeeping for waiting-time and match-rate outcomes.
        if was_unhoused and self.current_housing != "none":
            self.model.matches_this_step += 1
            self.model.waiting_time_sum_this_step += self.search_time
            self.total_search_time += self.search_time
            self.search_time = 0
            self.ever_housed = True
        elif self.current_housing == "none":
            self.search_time += 1

    def choose_housing(self):
        options = {}

        self.last_choice_utility_studio = None
        self.last_choice_utility_shared = None

        # Studio
        if self.model.available_studios > 0:
            effective_studio_cost = max(
                self.model.studio_rent - self.model.compute_studio_allowance(),
                0.0,
            )
            if effective_studio_cost <= self.income:
                u_studio = self.utility_studio(effective_studio_cost)
                options["studio"] = u_studio
                self.last_choice_utility_studio = u_studio

        # Shared
        if self.model.available_shared > 0:
            effective_shared_cost = self.model.shared_rent
            if effective_shared_cost <= self.income:
                u_shared = self.utility_shared(effective_shared_cost)
                options["shared"] = u_shared
                self.last_choice_utility_shared = u_shared

        # Forced-choice: if at least one real option exists, choose the best one.
        if options:
            return max(options, key=options.get)

        return "none"

    def utility_studio(self, cost: float):
        affordability = -cost / max(self.income, 1.0)
        preference_fit = 0.8 if self.housing_preference == "studio" else -0.6
        privacy_benefit = 0.6 * self.privacy_pref
        social_penalty = -0.6 * self.social_need
        scarcity_penalty = -0.20 * self.model.scarcity_studio
        time_bonus = 0.10 * self.time_pressure 

        return (
            1.25 * np.tanh(affordability * 3)
            + preference_fit
            + privacy_benefit
            + social_penalty
            + scarcity_penalty
            + time_bonus
        )

    def utility_shared(self, cost: float):
        affordability = -cost / max(self.income, 1.0)
        preference_fit = 0.8 if self.housing_preference == "shared" else -0.6
        social_benefit = 0.6 * self.social_need
        privacy_penalty = -0.6 * self.privacy_pref
        scarcity_penalty = -0.20 * self.model.scarcity_shared
        coordination_penalty = -0.08 * self.time_pressure

        return (
            1.25 * np.tanh(affordability * 3)
            + preference_fit
            + social_benefit
            + privacy_penalty
            + scarcity_penalty
            + coordination_penalty
        )

    def advance_wellbeing(self):
        # Loneliness update after the housing allocation.
        self.mismatch = 0.0 if self.current_housing == self.housing_preference else 1.0

        interaction_quality_shared = self.model.interaction_quality_shared
        interaction_quality_studio = self.model.interaction_quality_studio

        if self.current_housing == "studio":
            housing_cost = max(
                self.model.studio_rent - self.model.compute_studio_allowance(),
                0.0,
            )
            isolation = self.social_need
            interaction_quantity = float(np.clip(self.model.random.gauss(0.27, 0.12), 0.0, 1.0))
            routine_interaction = interaction_quality_studio * interaction_quantity

        elif self.current_housing == "shared":
            housing_cost = self.model.shared_rent
            isolation = self.privacy_pref
            interaction_quantity = float(np.clip(self.model.random.gauss(0.625, 0.12), 0.0, 1.0))
            routine_interaction = interaction_quality_shared * interaction_quantity

        else:
            housing_cost = self.income * 0.5
            isolation = 0.5
            routine_interaction = 0.02

        rent_burden = housing_cost / max(self.income, 1.0)
        stress = max(rent_burden - 0.35, 0.0)

        delta = (
            0.12 * self.loneliness_sensitivity * self.mismatch
            + 0.10 * isolation
            + 0.20 * self.stress_sensitivity * stress # Changed from 0.10 to 0.20 to show better the financial stress effect
            - 0.25 * routine_interaction # Changed from 0.12 to 0.25 to give a better role to interaction 
            - 0.03 * self.loneliness        # NEW : loneliness decay naturall over time, ref: Cacioppo & Hawkley (2010), who document that loneliness fluctuates around a personal baseline rather than accumulating indefinitely
        )

        self.loneliness = min(max(self.loneliness + delta, 0.0), 1.0)

### Housing model
Manages the housing stock, rents, allocation, landlord and developer behaviour, subsidy policy, and data collection. Each step: landlords withdraw/convert, developers build, students choose, rents adjust, and wellbeing updates.

In [ ]:
class StudentHousingModel(mesa.Model):
    # Housing market model using the student agents
    # Students make a forced short-term housing choice, 
    # rents adjust depending on supply and demand,
    #  developers build in response to market signals,
    # investors pull units out when they stop being profitable.

    def __init__(
        self,
        n_students=300,
        initial_studios=75,
        move_probability=0.25,
        initial_shared=129,
        studio_build_cost=486.0,
        shared_build_cost=347.0,
        studio_allowance_regime="none",
        studio_allowance_reduction=0.0,
        shared_rent_allowance=0.0,
        investor_withdrawal_rate=0.05,
        rent_adjustment_speed=0.017,
        vacancy_sensitivity=0.03,
        min_studio_rent=450.0,
        max_studio_rent=1500.0,
        min_shared_rent=420.0,
        max_shared_rent=1000.0,
        rent_inertia=0.2,
        operating_cost=500.0,
        coverage_ratio=0.90,
        conversion_rate=0.10,
        studio_rent_eq=750.0,
        shared_rent_eq=500.0,
        gamma_studio=0.03,
        gamma_shared=0.03,
        natural_vacancy_studio=0.03,
        natural_vacancy_shared=0.03,
        kappa_buffer_pct=0.20,
        studio_m2_per_person=25.0,
        shared_m2_per_person=12.0,
        shared_household_size=4,
        low_income_threshold=1000.0,
        privacy_lonely_threshold=0.7,
        construction_lag_ticks=24,
        crossfinance_enabled=False,
        object_subsidy_per_room=0.0,
        budget_cap_per_year=float("inf"),
        reference_gap=200.0,
        interaction_quality_studio=0.31,
        interaction_quality_shared=0.72,
        seed=42,
    ):
        super().__init__(seed=seed)

        # Seeded NumPy generator for trait draws (reproducible from `seed`).
        self.np_rng = np.random.default_rng(seed)

        # Stock
        self.studio_units = initial_studios
        self.shared_units = initial_shared

        # Prices
        self.studio_rent = studio_rent_eq
        self.shared_rent = shared_rent_eq

        self.min_studio_rent = min_studio_rent
        self.max_studio_rent = max_studio_rent
        self.min_shared_rent = min_shared_rent
        self.max_shared_rent = max_shared_rent

        self.rent_adjustment_speed = rent_adjustment_speed
        self.vacancy_sensitivity = vacancy_sensitivity
        self.rent_inertia = rent_inertia

        self.crossfinance_pool = 0.0

        # Landlord conversion
        self.coverage_ratio = coverage_ratio
        self.operating_cost = operating_cost
        self.min_income_threshold = coverage_ratio * self.operating_cost
        self.conversion_rate = conversion_rate

        # Equilibrium parameters
        self.studio_rent_eq = studio_rent_eq
        self.shared_rent_eq = shared_rent_eq
        self.natural_vacancy_studio = natural_vacancy_studio
        self.natural_vacancy_shared = natural_vacancy_shared

        # Cost structure
        self.studio_build_cost = studio_build_cost
        self.shared_build_cost = shared_build_cost

        # Developer trigger parameters
        self.kappa_buffer_pct = kappa_buffer_pct
        self.kappa_studio = (self.studio_build_cost - self.studio_rent_eq) + (kappa_buffer_pct * self.studio_build_cost)
        self.kappa_shared = (self.shared_build_cost - self.shared_rent_eq) + (kappa_buffer_pct * self.shared_build_cost)
        self.object_subsidy_per_room = object_subsidy_per_room
        self.kappa_shared_subsidy = (self.shared_build_cost - self.shared_rent_eq - (self.object_subsidy_per_room / 12 / 12 / 3)) + (kappa_buffer_pct * self.shared_build_cost)
        self.gamma_studio = gamma_studio
        self.gamma_shared = gamma_shared

        # Construction pipeline
        self.construction_lag_ticks = construction_lag_ticks
        self.construction_pipeline = {}
        self.step_count = 0

        # Policy
        self.crossfinance_enabled = crossfinance_enabled
        self.studio_allowance_regime = studio_allowance_regime
        self.studio_allowance_reduction = studio_allowance_reduction
        self.shared_rent_allowance = shared_rent_allowance
        self.investor_withdrawal_rate = investor_withdrawal_rate
        self.budget_cap_per_year = budget_cap_per_year
        self.subsidy_spent_this_year = 0.0
        self.cumulative_public_cost = 0.0

        # Tunable behavioural constants
        self.reference_gap = reference_gap
        self.interaction_quality_studio = interaction_quality_studio
        self.interaction_quality_shared = interaction_quality_shared

        # Outcome assumptions
        self.studio_m2_per_person = studio_m2_per_person
        self.shared_m2_per_person = shared_m2_per_person
        self.shared_household_size = shared_household_size
        self.low_income_threshold = low_income_threshold
        self.privacy_lonely_threshold = privacy_lonely_threshold

        # Dynamic counters
        self.available_studios = self.studio_units
        self.available_shared = self.shared_units

        self.scarcity_studio = 0.0
        self.scarcity_shared = 0.0

        self.realized_demand_studio = 0
        self.realized_demand_shared = 0
        self.realized_unhoused = 0
        self.seekers_this_step = 0
        self.matches_this_step = 0
        self.waiting_time_sum_this_step = 0

        self.last_vacancy_studio = 0.0
        self.last_vacancy_shared = 0.0

        self.last_withdrawn_studios = 0
        self.last_withdrawn_shared = 0
        self.last_built_studios = 0
        self.last_built_shared = 0

        # Build-trigger diagnostics: units queued by the developer this step.
        self.last_queued_studios = 0
        self.last_queued_shared = 0

        # Previous-step latent demand proxies
        self.latency_demand_studio = initial_studios
        self.latency_demand_shared = initial_shared

        self.move_probability = move_probability

        # Student incomes
        incomes = generate_student_incomes(
            n_agents=n_students,
            max_values=INCOME_DECILE_MAX_VALUES,
            mean_values=INCOME_DECILE_MEAN_VALUES,
            seed=seed,
        )

        # Create student agents (all trait draws use the seeded generator)
        for i in range(n_students):
            income = incomes[i]
            privacy_pref = float(np.clip(bimodal_sample(self.np_rng), 0.0, 1.0))   # private vs. open clustering
            social_need = float(np.clip(bimodal_sample(self.np_rng), 0.0, 1.0))    # introvert vs. extrovert split
            time_pressure = float(np.clip(beta_sample(self.np_rng), 0.0, 1.0))     # mostly low urgency, pressured tail
            loneliness_sensitivity = self.random.gauss(1.0, 0.2)
            stress_sensitivity = self.random.gauss(1.0, 0.2)
            # Beta(2,3): right-skewed, most students moderately price-sensitive.
            price_sensitivity = self.random.betavariate(2, 3)

            StudentAgent(
                self,
                income=income,
                privacy_pref=privacy_pref,
                social_need=social_need,
                time_pressure=time_pressure,
                loneliness_sensitivity=loneliness_sensitivity,
                stress_sensitivity=stress_sensitivity,
                price_sensitivity=price_sensitivity,
            )

        # Data collection
        self.datacollector = DataCollector(
            model_reporters={
                "effective_studio_cost": lambda m: max(m.studio_rent - m.compute_studio_allowance(), 0.0),
                "studio_units": lambda m: m.studio_units,
                "shared_units": lambda m: m.shared_units,
                "studio_rent": lambda m: m.studio_rent,
                "shared_rent": lambda m: m.shared_rent,
                "students_in_studio": lambda m: m.count_housing("studio"),
                "students_in_shared": lambda m: m.count_housing("shared"),
                "students_unhoused": lambda m: m.count_housing("none"),
                "share_matched": lambda m: m.share_matched(),
                "avg_mismatch": lambda m: m.avg_mismatch(),
                "avg_loneliness": lambda m: m.avg_loneliness(),
                "studio_allowance_cost": lambda m: m.studio_allowance_cost(),
                "total_public_cost": lambda m: m.total_public_cost(),
                "cumulative_public_cost": lambda m: m.cumulative_public_cost_metric(),
                "vacancy_studio": lambda m: m.last_vacancy_studio,
                "vacancy_shared": lambda m: m.last_vacancy_shared,
                "withdrawn_studios": lambda m: m.last_withdrawn_studios,
                "withdrawn_shared": lambda m: m.last_withdrawn_shared,
                "built_studios": lambda m: m.last_built_studios,
                "built_shared": lambda m: m.last_built_shared,
                "queued_studios": lambda m: m.last_queued_studios,
                "queued_shared": lambda m: m.last_queued_shared,
                "scarcity_studio": lambda m: m.scarcity_studio,
                "scarcity_shared": lambda m: m.scarcity_shared,

                # Outcome measures for scenario/grid comparison
                "avg_rent_per_person": lambda m: m.avg_rent_per_person(),
                "avg_rent_per_unit": lambda m: m.avg_rent_per_unit(),
                "avg_affordability_ratio": lambda m: m.avg_affordability_ratio(),
                "number_people_housed": lambda m: m.number_people_housed(),
                "total_vacancy_rate": lambda m: m.total_vacancy_rate(),
                "avg_waiting_time": lambda m: m.avg_waiting_time(),
                "match_rate": lambda m: m.match_rate(),
                "avg_space_per_person": lambda m: m.avg_space_per_person(),
                "landlord_revenue_total": lambda m: m.landlord_revenue_total(),
                "landlord_revenue_studio": lambda m: m.landlord_revenue_studio(),
                "landlord_revenue_shared": lambda m: m.landlord_revenue_shared(),
                "low_income_access_rate": lambda m: m.low_income_access_rate(),
                "social_contact_opportunities": lambda m: m.avg_social_contact_opportunities(),
                "privacy_lonely_share": lambda m: m.privacy_lonely_share(),
                "units_in_pipeline": lambda m: sum(v["studio"] + v["shared"] for v in m.construction_pipeline.values()),
                "subsidy_spent_this_year": lambda m: m.subsidy_spent_this_year,
                "subsidy_remaining":       lambda m: max(0, m.budget_cap_per_year - m.subsidy_spent_this_year),
            },
            agent_reporters={
                "housing_preference": "housing_preference",
                "current_housing": "current_housing",
                "mismatch": "mismatch",
                "loneliness": "loneliness",
                "income": "income",
                "search_time": "search_time",
                "social_contact_opportunities": lambda a: a.model.agent_social_contact_opportunities(a),
                "privacy_lonely_flag": lambda a: a.model.agent_privacy_lonely_flag(a),
            },
        )

        self.datacollector.collect(self)

    # Huurtoeslag calculation helper
    def compute_studio_allowance(self):
        # Dutch huurtoeslag for studios (2026 numbers from toeslagen.nl).
        # Covers 100% of rent between €202.52 and €498.20, 65% up to €713.02, then
        # 40% up to the max rent limit and 0% above it.
        # regime "none" -> no subsidy; "pre_2026" caps at €900.07; "post_2026" at €932.93.
        if self.studio_allowance_regime == "none":
            return 0.0

        MIN_BASIC_RENT       = 202.52
        QUALITY_LIMIT        = 498.20
        CAPPING_LIMIT        = 713.02
        MAX_RENT_PRE_2026    = 900.07
        MAX_RENT_POST_2026   = 932.93

        rent_cap = (
            MAX_RENT_PRE_2026
            if self.studio_allowance_regime == "pre_2026"
            else MAX_RENT_POST_2026
        )

        if self.studio_rent > rent_cap:
            return 0.0
        if self.studio_rent <= MIN_BASIC_RENT:
            return 0.0

        allowance = 0.0

        band1_top = min(self.studio_rent, QUALITY_LIMIT)
        if band1_top > MIN_BASIC_RENT:
            allowance += 1.00 * (band1_top - MIN_BASIC_RENT)

        if self.studio_rent > QUALITY_LIMIT:
            band2_top = min(self.studio_rent, CAPPING_LIMIT)
            allowance += 0.65 * (band2_top - QUALITY_LIMIT)

        if self.studio_rent > CAPPING_LIMIT:
            band3_top = min(self.studio_rent, rent_cap)
            allowance += 0.40 * (band3_top - CAPPING_LIMIT)

        return max(allowance - self.studio_allowance_reduction, 0.0)

    # MAIN STEP

    def step(self):
        # Advance the clock once per step.
        self.step_count += 1

        # Reset annual subsidy budget at the start of each new year.
        if self.step_count % 12 == 0:
            self.subsidy_spent_this_year = 0.0
            # self.crossfinance_pool = 0.0   # ← Do I want the cross finance budget to reset? 

        # Reset monthly counters
        self.realized_demand_studio = 0
        self.realized_demand_shared = 0
        self.realized_unhoused = 0
        self.seekers_this_step = 0
        self.matches_this_step = 0
        self.waiting_time_sum_this_step = 0
        self.last_withdrawn_studios = 0
        self.last_withdrawn_shared = 0
        self.last_built_studios = 0
        self.last_built_shared = 0
        self.last_queued_studios = 0
        self.last_queued_shared = 0

        # 1. Landlords withdraw unprofitable units
        self.landlord_withdrawal()

        # 2. Developers build based on expected profitability
        self.developer_build_decision()

        # 3. Release units that have completed construction
        self.release_construction_pipeline()

        # 4. Only vacant units are available for this step's allocation
        occupied_studios = self.count_housing("studio")
        occupied_shared = self.count_housing("shared")
        self.available_studios = max(self.studio_units - occupied_studios, 0)
        self.available_shared = max(self.shared_units - occupied_shared, 0)

        # 5. Students choose housing in random order
        self.agents.shuffle_do("step")

        # 6. Update scarcity and vacancy
        self.update_market_state()

        # 7. Endogenous rent adjustment
        self.adjust_rents()

        # 8. Update wellbeing
        for agent in self.agents:
            agent.advance_wellbeing()

        # 9. Update cumulative public spending
        self.cumulative_public_cost += self.studio_allowance_cost()

        self.cumulative_public_cost += (self.count_housing("shared") * self.shared_rent_allowance)

        # 10. Collect data
        self.datacollector.collect(self)

    # MARKET DYNAMICS

    def landlord_withdrawal(self):
        """Landlord decision-making: convert between types or exit market."""
        occupancy_studio = self.count_housing("studio") / max(self.studio_units, 1)
        occupancy_shared = self.count_housing("shared") / max(self.shared_units, 1)

        income_studio = self.studio_rent * occupancy_studio
        income_shared = self.shared_rent * occupancy_shared

        if max(income_studio, income_shared) < self.min_income_threshold:
            if self.studio_units > 0:
                removed = max(1, int(self.studio_units * self.investor_withdrawal_rate))
                self.studio_units = max(self.studio_units - removed, 0)
                self.last_withdrawn_studios = removed

            if self.shared_units > 0:
                removed = max(1, int(self.shared_units * self.investor_withdrawal_rate))
                self.shared_units = max(self.shared_units - removed, 0)
                self.last_withdrawn_shared = removed
            return

        if income_studio > income_shared:
            # POLICY GUARD: do not convert shared→studio under active shared room subsidy
            # Reflects that subsidised landlords are contractually committed to shared use
            # (RVO, 2025; Schilder & Conijn, 2015)
            subsidy_active = (
                getattr(self, "object_subsidy_per_room", 0) > 0
                or getattr(self, "crossfinance_enabled", False)
            )
            if not subsidy_active:
                convert = max(1, int(self.shared_units * self.conversion_rate))
                convert = min(convert, self.shared_units)
                self.shared_units -= convert
                self.studio_units += convert

        elif income_shared > income_studio:   
            convert = max(1, int(self.studio_units * self.conversion_rate))
            convert = min(convert, self.studio_units)
            self.studio_units -= convert
            self.shared_units += convert

    def developer_build_decision(self):
        """
        Developers build based on rent and vacancy rate.

        Policy extension:
            - object_subsidy: fixed government budget subsidises shared room construction
            - platform31_crossfinance: budget is dynamically pooled from huurtoeslag
              savings (studios_receiving_allowance × studio_allowance_reduction)
              rather than a fixed allocation. Same subsidy mechanism, different
              budget source.
        """
        # --- POLICY: compute available subsidy budget this step ---
        if self.crossfinance_enabled:
            # NOTE: compute_studio_allowance() is a model-level method (the allowance
            # depends on studio_rent + regime, not per-agent), so it is called on
            # self, not on the agent.
            studios_receiving_allowance = sum(
                1 for agent in self.agents
                if getattr(agent, "current_housing", None) == "studio"
                and self.compute_studio_allowance() > 0
            )
            monthly_savings = (
            studios_receiving_allowance
            * self.studio_allowance_reduction
            )
            # Accumulate savings into the annual pool (same logic as
            # subsidy_spent_this_year, but building up rather than spending down)
            self.crossfinance_pool += monthly_savings
            available_budget = max(0, self.crossfinance_pool - self.subsidy_spent_this_year)
            subsidy_spent = 0

        elif self.object_subsidy_per_room > 0:
            # Fixed annual budget — remaining = cap minus what's been spent this year
            available_budget = max(
                0, self.budget_cap_per_year - self.subsidy_spent_this_year
            )
            subsidy_spent = 0

        else:
            available_budget = 0
            subsidy_spent = 0

        for btype in ["studio", "shared"]:
            if btype == "studio":
                rent = self.studio_rent
                rent_eq = self.studio_rent_eq
                kappa = self.kappa_studio
                gamma = self.gamma_studio
                vacancy = self.last_vacancy_studio
                natural_vac = self.natural_vacancy_studio
            else:
                rent = self.shared_rent
                rent_eq = self.shared_rent_eq
                kappa = self.kappa_shared
                gamma = self.gamma_shared
                vacancy = self.last_vacancy_shared
                natural_vac = self.natural_vacancy_shared

            # --- POLICY: apply object subsidy to shared rooms only ---
            # Subsidy reduces effective kappa (the build cost threshold),
            # making construction of shared rooms more attractive relative
            # to studios. Subsidy is expressed in model rent units by
            # amortising over 12 years × 12 months (see build cost derivation).

            if btype == "shared" and self.object_subsidy_per_room > 0:
                effective_kappa = self.kappa_shared_subsidy 
            else:
                effective_kappa = kappa

            rent_signal = rent - rent_eq
            vacancy_signal = natural_vac - vacancy

            if rent_signal > effective_kappa and vacancy_signal > 0:
                new_units = int(gamma * rent_signal * vacancy_signal)
                new_units = max(1, new_units)

                # --- POLICY: check available budget before building ---
                if btype == "shared" and self.object_subsidy_per_room > 0:
                    subsidy_cost = new_units * self.object_subsidy_per_room
                    if subsidy_spent + subsidy_cost > available_budget:
                        affordable_units = int(
                            (available_budget - subsidy_spent)
                            / max(self.object_subsidy_per_room, 1)
                        )
                        new_units = max(0, affordable_units)
                    subsidy_spent += new_units * self.object_subsidy_per_room
                    object_cost = new_units * self.object_subsidy_per_room

                    self.subsidy_spent_this_year += object_cost
                    self.cumulative_public_cost += object_cost

                if new_units > 0:
                    # Build-trigger diagnostics
                    if btype == "studio":
                        self.last_queued_studios += new_units
                    else:
                        self.last_queued_shared += new_units

                    delivery_step = self.step_count + self.construction_lag_ticks
                    if delivery_step not in self.construction_pipeline:
                        self.construction_pipeline[delivery_step] = {"studio": 0, "shared": 0}
                    self.construction_pipeline[delivery_step][btype] += new_units

    def release_construction_pipeline(self):
        """Release units from the pipeline when their delivery step is reached."""
        if self.step_count in self.construction_pipeline:
            delivery = self.construction_pipeline.pop(self.step_count)

            studios_delivered = delivery.get("studio", 0)
            shared_delivered = delivery.get("shared", 0)

            self.studio_units += studios_delivered
            self.shared_units += shared_delivered

            self.last_built_studios += studios_delivered
            self.last_built_shared += shared_delivered

    def update_market_state(self):
        """Update scarcity, vacancy, and latent demand proxies after allocation."""
        occupied_studios = self.count_housing("studio")
        occupied_shared = self.count_housing("shared")

        self.last_vacancy_studio = max(self.studio_units - occupied_studios, 0) / max(self.studio_units, 1)
        self.last_vacancy_shared = max(self.shared_units - occupied_shared, 0) / max(self.shared_units, 1)

        unmet_studio = sum(
            1 for a in self.agents
            if a.housing_preference == "studio" and a.current_housing != "studio"
        )
        unmet_shared = sum(
            1 for a in self.agents
            if a.housing_preference == "shared" and a.current_housing != "shared"
        )

        self.scarcity_studio = unmet_studio / max(self.studio_units, 1)
        self.scarcity_shared = unmet_shared / max(self.shared_units, 1)

        self.latency_demand_studio = occupied_studios + unmet_studio
        self.latency_demand_shared = occupied_shared + unmet_shared

    def adjust_rents(self):
        """Endogenous rent update: rises with scarcity, falls with vacancy."""
        studio_growth = (
            self.rent_adjustment_speed * self.scarcity_studio
            - self.vacancy_sensitivity * self.last_vacancy_studio
        )
        shared_growth = (
            self.rent_adjustment_speed * self.scarcity_shared
            - self.vacancy_sensitivity * self.last_vacancy_shared
        )

        target_studio_rent = self.studio_rent * (1.0 + studio_growth)
        target_shared_rent = self.shared_rent * (1.0 + shared_growth)

        inertia = self.rent_inertia
        self.studio_rent = (1 - inertia) * self.studio_rent + inertia * target_studio_rent
        self.shared_rent = (1 - inertia) * self.shared_rent + inertia * target_shared_rent

        self.studio_rent = min(max(self.studio_rent, self.min_studio_rent), self.max_studio_rent)
        self.shared_rent = min(max(self.shared_rent, self.min_shared_rent), self.max_shared_rent)

    # METRICS

    def count_housing(self, housing_type):
        return sum(1 for a in self.agents if a.current_housing == housing_type)

    def share_matched(self):
        return sum(1 for a in self.agents if a.current_housing == a.housing_preference) / max(len(self.agents), 1)

    def avg_mismatch(self):
        return sum(a.mismatch for a in self.agents) / max(len(self.agents), 1)

    def avg_loneliness(self):
        return sum(a.loneliness for a in self.agents) / max(len(self.agents), 1)

    def housing_cost_for_agent(self, agent):
        if agent.current_housing == "studio":
            return max(self.studio_rent - self.compute_studio_allowance(), 0.0)
        if agent.current_housing == "shared":
            return max(self.shared_rent - self.shared_rent_allowance, 0.0)
        return 0.0

    def number_people_housed(self):
        return self.count_housing("studio") + self.count_housing("shared")

    def avg_rent_per_person(self):
        housed = [a for a in self.agents if a.current_housing != "none"]
        if not housed:
            return 0.0
        return sum(self.housing_cost_for_agent(a) for a in housed) / len(housed)

    def avg_rent_per_unit(self):
        occupied_units = self.number_people_housed()
        if occupied_units == 0:
            return 0.0
        return self.landlord_revenue_total() / occupied_units

    def avg_affordability_ratio(self):
        housed = [a for a in self.agents if a.current_housing != "none"]
        if not housed:
            return 0.0
        return sum(self.housing_cost_for_agent(a) / max(a.income, 1.0) for a in housed) / len(housed)

    def total_vacancy_rate(self):
        vacant = max(self.studio_units - self.count_housing("studio"), 0) + max(self.shared_units - self.count_housing("shared"), 0)
        total_units = self.studio_units + self.shared_units
        return vacant / max(total_units, 1)

    def avg_waiting_time(self):
        if self.matches_this_step == 0:
            return 0.0
        return self.waiting_time_sum_this_step / self.matches_this_step

    def match_rate(self):
        return self.matches_this_step / max(self.seekers_this_step, 1)

    def avg_space_per_person(self):
        housed = [a for a in self.agents if a.current_housing != "none"]
        if not housed:
            return 0.0
        total_space = (
            self.count_housing("studio") * self.studio_m2_per_person
            + self.count_housing("shared") * self.shared_m2_per_person
        )
        return total_space / len(housed)

    def landlord_revenue_studio(self):
        return self.count_housing("studio") * self.studio_rent

    def landlord_revenue_shared(self):
        return self.count_housing("shared") * self.shared_rent

    def landlord_revenue_total(self):
        return self.landlord_revenue_studio() + self.landlord_revenue_shared()

    def low_income_access_rate(self):
        low_income = [a for a in self.agents if a.income <= self.low_income_threshold]
        if not low_income:
            return 0.0
        return sum(1 for a in low_income if a.current_housing != "none") / len(low_income)

    def agent_social_contact_opportunities(self, agent):
        if agent.current_housing == "shared":
            return max(self.shared_household_size - 1, 0) * (1.0 - agent.privacy_pref)
        return 0.0

    def avg_social_contact_opportunities(self):
        housed = [a for a in self.agents if a.current_housing != "none"]
        if not housed:
            return 0.0
        return sum(self.agent_social_contact_opportunities(a) for a in housed) / len(housed)

    def agent_privacy_lonely_flag(self, agent):
        return int(agent.current_housing == "studio" and agent.privacy_pref >= self.privacy_lonely_threshold)

    def privacy_lonely_share(self):
        housed = [a for a in self.agents if a.current_housing != "none"]
        if not housed:
            return 0.0
        return sum(self.agent_privacy_lonely_flag(a) for a in housed) / len(housed)

    def studio_allowance_cost(self):
        return self.count_housing("studio") * self.compute_studio_allowance()

    def total_public_cost(self):     # how much is the government spending at each step
        return self.studio_allowance_cost()
    
    def cumulative_public_cost_metric(self):   # how much is the government spending in total 
        return self.cumulative_public_cost

### Single simulation run function

In [ ]:
def run_model(steps=40, **model_kwargs):
    # all keyword arguments are passed to the model init, so overrides can be defined
    model = StudentHousingModel(**model_kwargs)

    for _ in range(steps):
        model.step()

    model_df = model.datacollector.get_model_vars_dataframe()
    agent_df = model.datacollector.get_agent_vars_dataframe()
    return model, model_df, agent_df

### Plotting & diagnostics helpers
Defined here so every later section can use them. They are used in the baseline, scenario, and sensitivity sections below.

In [ ]:
def plot_trajectory_band(trajectories, outcome="avg_loneliness", scenarios=None):
    # Mean ± SD across seeds over time, one band per scenario.
    plt.figure(figsize=(11, 6))
    for name, seed_dfs in trajectories.items():
        if scenarios and name not in scenarios:
            continue
        mat = pd.concat([df[outcome] for df in seed_dfs.values()], axis=1)
        mean = mat.mean(axis=1)
        std = mat.std(axis=1)
        line, = plt.plot(mean.index, mean.values, label=name)
        plt.fill_between(mean.index, (mean - std).values, (mean + std).values,
                         alpha=0.15, color=line.get_color())
    plt.xlabel("Step")
    plt.ylabel(outcome)
    plt.title(f"{outcome.replace('_', ' ').title()} (mean ± SD across seeds)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


def plot_outcome_distributions(tidy, outcome="avg_loneliness"):
    # Boxplot of an outcome across seeds, one box per scenario.
    order = tidy.groupby("scenario")[outcome].mean().sort_values().index.tolist()
    data = [tidy.loc[tidy["scenario"] == s, outcome].values for s in order]
    plt.figure(figsize=(11, 6))
    plt.boxplot(data, tick_labels=order, vert=False, showmeans=True)
    plt.xlabel(outcome)
    plt.title(f"{outcome.replace('_', ' ').title()} across seeds")
    plt.tight_layout()
    plt.show()


def plot_supply_trajectory(model_df, title="Housing supply"):
    plt.figure(figsize=(10, 6))
    plt.plot(model_df.index, model_df["studio_units"], label="Studio units")
    plt.plot(model_df.index, model_df["shared_units"], label="Shared units")
    plt.xlabel("Step")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_metric(results_df, metric):
    plt.figure(figsize=(10, 6))
    ordered = results_df.sort_values(metric)
    plt.barh(ordered["scenario"], ordered[metric])
    plt.xlabel(metric)
    plt.ylabel("Scenario")
    plt.title(metric.replace("_", " ").title())
    plt.tight_layout()
    plt.show()

In [ ]:
def build_trigger_report(trajectories):
    # Per scenario: how many units got queued (build triggered) and delivered, per seed.
    rows = []
    for name, seed_dfs in trajectories.items():
        queued = delivered = 0.0
        for df in seed_dfs.values():
            for col in ("queued_studios", "queued_shared"):
                if col in df:
                    queued += float(df[col].sum())
            for col in ("built_studios", "built_shared"):
                if col in df:
                    delivered += float(df[col].sum())
        n = max(len(seed_dfs), 1)
        rows.append({
            "scenario": name,
            "units_queued_per_seed": queued / n,
            "units_delivered_per_seed": delivered / n,
            "build_trigger_ever_fired": queued > 0,
        })
    return pd.DataFrame(rows).set_index("scenario")

## 6. Baseline run and sanity checks

Before comparing any policies I run the model once with nothing switched on, just to check that supply, rents, and loneliness do something believable.

In [ ]:
# One baseline scenario run, to confirm model behaves as expected
baseline_model, baseline_df, baseline_agents = run_model(
    steps=RUN_CONFIG["steps"], **{**MODEL_PARAMS, "seed": 42}
)

print("Final-step snapshot:")
for k in ["studio_units", "shared_units", "studio_rent", "shared_rent",
          "students_in_studio", "students_in_shared", "students_unhoused",
          "share_matched", "avg_loneliness"]:
    print(f"  {k:22s} {baseline_df.iloc[-1][k]:.3f}")

In [ ]:
# Quick sanity plots: do supply, rents, and loneliness move in a believable way?
plot_supply_trajectory(baseline_df, title="Baseline housing supply over time")

for outcome in ["studio_rent", "shared_rent", "avg_loneliness", "total_vacancy_rate"]:
    plt.figure(figsize=(9, 3.2))
    plt.plot(baseline_df.index, baseline_df[outcome])
    plt.xlabel("Step (month)")
    plt.ylabel(outcome)
    plt.title(outcome.replace("_", " ").title())
    plt.tight_layout()
    plt.show()

## 7. Policy scenarios

Each scenario defines variables that deviate from `MODEL_PARAMS`. Scenarios can be named at the first level and vairables (must have been defined earlier) added or edited within them

In [ ]:
def scenario_dict():

    return {

        "baseline_no_subsidy": {
            "studio_allowance_regime": "none",
            "shared_rent_allowance": 0,
            "studio_allowance_reduction": 0,
            "object_subsidy_per_room": 0,
            "crossfinance_enabled": False,
        },

        "pre_2026": {
            # max huurtoeslag ≈ €498, rent cap €900.07 (handled in compute_studio_allowance)
            "studio_allowance_regime": "pre_2026",
            "shared_rent_allowance": 0,
            "studio_allowance_reduction": 0,
            "object_subsidy_per_room": 0,
            "crossfinance_enabled": False,
        },

        "post_2026": {
            # max huurtoeslag ≈ €523, hard rent cap removed (cap €932.93 in model)
            "studio_allowance_regime": "post_2026",
            "shared_rent_allowance": 0,
            "studio_allowance_reduction": 0,
            "object_subsidy_per_room": 0,
            "crossfinance_enabled": False,
        },

        "studio_reduction": {
            "studio_allowance_regime": "post_2026",
            "studio_allowance_reduction": 150,
            "shared_rent_allowance": 0,
            "object_subsidy_per_room": 0,
            "crossfinance_enabled": False,
        },

        "room_subsidy": {
            "studio_allowance_regime": "post_2026",
            "studio_allowance_reduction": 0,
            "shared_rent_allowance": 200,
            "object_subsidy_per_room": 0,
            "crossfinance_enabled": False,
        },

        "demand_combined": {
            "studio_allowance_regime": "post_2026",
            "studio_allowance_reduction": 150,
            "shared_rent_allowance": 200,
            "object_subsidy_per_room": 0,
            "crossfinance_enabled": False,
        },

        "object_subsidy": {
            "studio_allowance_regime": "post_2026",
            "studio_allowance_reduction": 0,
            "shared_rent_allowance": 0,
            "object_subsidy_per_room": 40000,
            "budget_cap_per_year": 500000,  # midpoint of cross-finance implied range (144k–200k)
            # "construction_lag_ticks": 24,
            "crossfinance_enabled": False,
        },

        "platform31_crossfinance": {
            "studio_allowance_regime": "post_2026",
            "studio_allowance_reduction": 150,
            "shared_rent_allowance": 0,
            "object_subsidy_per_room": 40000,
            # "construction_lag_ticks": 24,
            "crossfinance_enabled": True,
        },

        "full_intervention": {
            "studio_allowance_regime": "post_2026",
            "studio_allowance_reduction": 150,
            "shared_rent_allowance": 200,
            "object_subsidy_per_room": 40000,
            # "construction_lag_ticks": 24,
            "crossfinance_enabled": True,
        },
    }

### Quick single-seed comparison
Later we will re-run this with more random seeds to get the 'general' run pattern without major stochastic effects.

In [ ]:
def compare_scenarios(steps=40, seed=42, base_kwargs=None):
    base_kwargs = base_kwargs or {}
    rows = []

    for scenario_name, params in scenario_dict().items():
        _, model_df, _ = run_model(steps=steps, **{**base_kwargs, **params, "seed": seed})

        final = model_df.iloc[-1].to_dict()
        final["scenario"] = scenario_name
        final["studio_allowance_regime"] = params.get("studio_allowance_regime", "none")
        final["shared_rent_allowance"] = params.get("shared_rent_allowance", 0)

        rows.append(final)

    if not rows:
        raise ValueError("No scenarios found. Is scenario_dict() returning an empty dict?")

    comparison = pd.DataFrame(rows)

    cols = [
        "scenario", "studio_allowance_regime", "shared_rent_allowance",
        "studio_units", "shared_units", "studio_rent", "shared_rent",
        "effective_studio_cost", "students_in_studio", "students_in_shared",
        "students_unhoused", "share_matched", "avg_mismatch", "avg_loneliness",
        "total_public_cost", "cumulative_public_cost", "avg_rent_per_person", "avg_affordability_ratio",
        "number_people_housed", "total_vacancy_rate", "avg_waiting_time",
        "match_rate", "avg_space_per_person", "landlord_revenue_total",
        "low_income_access_rate", "social_contact_opportunities", "privacy_lonely_share",
    ]
    return comparison[[c for c in cols if c in comparison.columns]]

In [ ]:
# generate a comparison table of all scenarios, using a single random seed - multi-seed averages follow later
single_seed_table = compare_scenarios(
    steps=RUN_CONFIG["steps"], seed=42, base_kwargs=MODEL_PARAMS
)
single_seed_table.round(3)

## 8. Removing stochastic noise - multi-seed averaging

Each scenario gets run across a bunch of seeds and summarised by the average over the final steps. Each scenario average tracks a number of outcomes

In [ ]:
# DEFINITIONS
# 
# outcome families are used to group related metrics for reporting and plotting.
OUTCOME_FAMILIES = {
    "loneliness": [
        "avg_loneliness", "avg_mismatch", "share_matched",
        "privacy_lonely_share", "social_contact_opportunities",
    ],
    "access_affordability": [
        "avg_affordability_ratio", "avg_rent_per_person", "low_income_access_rate",
        "number_people_housed", "avg_waiting_time", "match_rate", "avg_space_per_person",
    ],
    "cost_supply": [
        "total_public_cost", "cumulative_public_cost", "studio_units", "shared_units", "total_vacancy_rate",
        "landlord_revenue_total", "units_in_pipeline",
        "queued_studios", "queued_shared", "built_studios", "built_shared",
    ],
}
ALL_OUTCOMES = [c for cols in OUTCOME_FAMILIES.values() for c in cols]


def _outcome_row(model_df, scenario, seed, window=50):
    # Summarise one run by the mean of each outcome over its last `window` steps.
    tail = model_df.tail(min(window, len(model_df)))
    row = {"scenario": scenario, "seed": seed}
    for c in ALL_OUTCOMES:
        if c in model_df.columns:
            row[c] = tail[c].mean()
    return row


def run_scenarios_multiseed(scenarios, steps=300, seeds=range(20), window=50, base_kwargs=None):
    # Run every scenario across several seeds.
    # Returns:
    #   trajectories = {scenario: {seed: model_df}}  (full time series per run)
    #   tidy = one row per (scenario, seed) with the end-window mean of every outcome.
    # The tidy table is what the summary / effect / ablation functions below use.
    base_kwargs = base_kwargs or {}
    trajectories = {}
    rows = []
    for name, params in scenarios.items():
        trajectories[name] = {}
        for seed in seeds:
            _, model_df, _ = run_model(steps=steps, **{**base_kwargs, **params, "seed": int(seed)})
            trajectories[name][int(seed)] = model_df
            rows.append(_outcome_row(model_df, name, int(seed), window=window))
    return trajectories, pd.DataFrame(rows)


def summarize_across_seeds(tidy, outcomes=None):
    # Per-scenario mean/std/min/max across seeds for each outcome.
    outcomes = outcomes or [c for c in tidy.columns if c not in ("scenario", "seed")]
    return tidy.groupby("scenario")[outcomes].agg(["mean", "std", "min", "max"])


def summarize_by_family(tidy):
    # One per-scenario (mean, std) table per outcome family, for reading trade-offs.
    out = {}
    for fam, cols in OUTCOME_FAMILIES.items():
        present = [c for c in cols if c in tidy.columns]
        if present:
            out[fam] = tidy.groupby("scenario")[present].agg(["mean", "std"])
    return out


def effect_vs_baseline(tidy, baseline="baseline_no_subsidy", outcome="avg_loneliness"):
    # Mean effect of each scenario vs the baseline on one outcome, with a noise flag.
    # separated_from_baseline is True when the scenario's [mean-std, mean+std] band
    # doesn't overlap the baseline's. It's a rough "is this beyond run-to-run noise?"
    # check for eyeballing, not a proper significance test.
    g = tidy.groupby("scenario")[outcome].agg(["mean", "std"])
    if baseline not in g.index:
        raise KeyError(f"baseline '{baseline}' not in results")
    b_mean, b_std = g.loc[baseline, "mean"], g.loc[baseline, "std"]
    b_lo, b_hi = b_mean - b_std, b_mean + b_std

    rows = []
    for sc, r in g.iterrows():
        lo, hi = r["mean"] - r["std"], r["mean"] + r["std"]
        separated = (hi < b_lo) or (lo > b_hi)
        rows.append({
            "scenario": sc,
            "mean": r["mean"],
            "std": r["std"],
            "delta_vs_baseline": r["mean"] - b_mean,
            "separated_from_baseline": bool(separated),
        })
    return pd.DataFrame(rows).set_index("scenario").sort_values("delta_vs_baseline")

### Run the evaluation

In [ ]:
# Multi-seed run across all the policy scenarios.
# steps / seeds come from RUN_CONFIG and the shared baseline comes from MODEL_PARAMS
# (both set in the Control Panel at the top).
trajectories, tidy = run_scenarios_multiseed(
    scenario_dict(),
    steps=RUN_CONFIG["steps"],
    seeds=RUN_CONFIG["seeds"],
    window=RUN_CONFIG["window"],
    base_kwargs=MODEL_PARAMS,
)

# (a) per-scenario outcome vectors with spread, grouped by family
family_tables = summarize_by_family(tidy)
for fam, table in family_tables.items():
    print(f"\n=== {fam.upper()} ===")
    print(table.round(3))

# effect on the headline outcome vs baseline, with the beyond-noise flag
loneliness_effect = effect_vs_baseline(tidy, outcome="avg_loneliness")
print("\n=== LONELINESS EFFECT vs BASELINE ===")
print(loneliness_effect.round(4))

# did any supply-side builds actually trigger?
print("\n=== BUILD-TRIGGER DIAGNOSTICS ===")
print(build_trigger_report(trajectories).round(2))

## 9. Marginal contribution of each policy lever

Starting from the full intervention, switch off one lever at a time to see how much each lever contributes to each outcome.

In [ ]:
# Levers in full_intervention and the value that counts as "off".
ABLATION_LEVERS = {
    "studio_allowance_reduction": 0,
    "shared_rent_allowance": 0,
    "object_subsidy_per_room": 0,
    "crossfinance_enabled": False,
}


def make_ablation_scenarios(base_name="full_intervention", levers=None):
    # Return {base, full_minus_<lever>, ...}, each one resetting exactly one lever.
    levers = levers or ABLATION_LEVERS
    base = dict(scenario_dict()[base_name])
    scenarios = {base_name: base}
    for lever, off_value in levers.items():
        if lever in base:
            variant = dict(base)
            variant[lever] = off_value
            scenarios[f"full_minus_{lever}"] = variant
    return scenarios


def marginal_contributions(tidy, base_name="full_intervention", outcomes=None):
    # outcome(full) - outcome(full_minus_lever) per lever, averaged across seeds.
    # Big positive value -> removing the lever pushes the outcome up, so the lever
    # was holding it down (and the other way round).
    outcomes = outcomes or [
        "avg_loneliness", "avg_affordability_ratio", "low_income_access_rate",
        "number_people_housed", "total_public_cost", "cumulative_public_cost"
    ]
    outcomes = [o for o in outcomes if o in tidy.columns]
    means = tidy.groupby("scenario")[outcomes].mean()
    if base_name not in means.index:
        raise KeyError(f"'{base_name}' not in results")

    rows = []
    for sc in means.index:
        if sc == base_name:
            continue
        lever = sc.replace("full_minus_", "")
        delta = means.loc[base_name] - means.loc[sc]
        rows.append({"removed_lever": lever, **delta.to_dict()})
    return pd.DataFrame(rows).set_index("removed_lever")

## 10. One-at-a-time sensitivity

Each parameter is varied across a range while keeping others constant to see how outcomes change.

In [ ]:
def run_oat_sensitivity(param, values, scenario_name, steps, seeds , window, base_kwargs=None):
    base = {**(base_kwargs or MODEL_PARAMS), **scenario_dict()[scenario_name]}
    rows = []
    for v in values:
        for seed in seeds:
            _, model_df, _ = run_model(steps=steps, **{**base, param: v, "seed": int(seed)})
            r = _outcome_row(model_df, f"{param}={v}", int(seed), window=window)
            r[param] = v
            rows.append(r)
    return pd.DataFrame(rows)


def sensitivity_summary(oat_df, param, outcome="avg_loneliness"):
    # For each parameter value: mean +/- std of an outcome across seeds.
    return oat_df.groupby(param)[outcome].agg(["mean", "std", "min", "max"])

### Test sweep

In [ ]:
SWEEP_PARAM = "shared_rent_allowance"
SWEEP_VALUES = [0, 100, 200, 300, 400]

oat_df = run_oat_sensitivity(
    param=SWEEP_PARAM,
    values=SWEEP_VALUES,
    scenario_name="post_2026",
    steps=RUN_CONFIG["steps"],
    seeds=range(5),
    window=RUN_CONFIG["window"],
)

for outcome in ["avg_loneliness", "cumulative_public_cost", "number_people_housed"]:
    print(f"\n=== {outcome} vs {SWEEP_PARAM} ===")
    print(sensitivity_summary(oat_df, SWEEP_PARAM, outcome=outcome).round(3))

In [ ]:
# Plot the sweep: mean outcome with a +/-SD band across seeds.
for outcome in ["avg_loneliness", "cumulative_public_cost"]:
    stats = sensitivity_summary(oat_df, SWEEP_PARAM, outcome=outcome)
    plt.figure(figsize=(8, 4))
    plt.plot(stats.index, stats["mean"], marker="o")
    plt.fill_between(stats.index,
                     stats["mean"] - stats["std"],
                     stats["mean"] + stats["std"], alpha=0.2)
    plt.xlabel(SWEEP_PARAM)
    plt.ylabel(outcome)
    plt.title(f"{outcome.replace('_', ' ').title()} vs {SWEEP_PARAM}")
    plt.tight_layout()
    plt.show()

## 11. Result plots

average outcome trajectories with the seed spread in shaded bands: the spread of loneliness across seeds, and the lever removal, using plotting helpers defined earlier in the notebook.

Note: Pre-2026 tracks exactly the post-2026 plot and therefore seems missing. It's the same plot.

### Trajectory bands across scenarios

In [ ]:
# Trajectory bands (mean +/- SD across seeds) for the main outcomes.
for outcome in ["avg_loneliness", "avg_affordability_ratio", "low_income_access_rate", "studio_rent", "shared_rent", "subsidy_spent_this_year", "cumulative_public_cost", "avg_mismatch"]:
    plot_trajectory_band(trajectories, outcome=outcome)

# spread of the headline outcome across seeds
plot_outcome_distributions(tidy, outcome="avg_loneliness")

### Lever ablation

In [ ]:
# Ablation: marginal contribution of each lever in full_intervention.
ablation_scenarios = make_ablation_scenarios("full_intervention")
ablation_traj, ablation_tidy = run_scenarios_multiseed(
    ablation_scenarios,
    steps=RUN_CONFIG["steps"],
    seeds=RUN_CONFIG["seeds"],
    window=RUN_CONFIG["window"],
    base_kwargs=MODEL_PARAMS,
)
print("\n=== MARGINAL CONTRIBUTIONS (full minus each lever) ===")
print(marginal_contributions(ablation_tidy).round(4))

### Scarcity, rents, and supply by scenario

In [ ]:
for outcome in ["scarcity_studio", "scarcity_shared"]:
    plot_trajectory_band(
        trajectories,
        outcome=outcome,
        scenarios=["baseline_no_subsidy", "studio_reduction", "room_subsidy", "object_subsidy", "platform31_crossfinance"]   # baseline just for reference
    )

# rents over time
for outcome in ["studio_rent", "shared_rent"]:
    plot_trajectory_band(
        trajectories,
        outcome=outcome,
        scenarios=["baseline_no_subsidy", "studio_reduction", "room_subsidy", "object_subsidy", "platform31_crossfinance"]
    )

for outcome in ["studio_units", "shared_units"]:
    plot_trajectory_band(trajectories, outcome=outcome,
                        scenarios=["baseline_no_subsidy", "object_subsidy", "platform31_crossfinance"])

## 12. Does the ranking hold?

This section tests the robustness of our policies randkings by varying key structural parameters, re-running all scenarios, and checking whether the policy ordering changes. Each parameter value is applied to every scenario, ensuring that only the policy differs.

Parameters tested:

* **`n_students`** – housing demand.
* **`construction_lag_ticks`** – speed of new supply.
* **`reference_gap`** – sensitivity to rent differences.
* **`interaction_quality_shared`** – social benefits of shared housing.

For each sweep, I report whether the top-ranked scenario changes and the rank correlation with the baseline ordering (`1.0` = identical ranking).


### The ranking machinery

In [ ]:
# Robustness of the scenario RANKING, not just the point values.
# The question: if I change a structural parameter, do the scenarios keep the same
# order on the key outcomes? A conclusion that only holds at one setting is fragile.

# Which direction is "better" for each outcome (rank 1 = best).
OUTCOME_DIRECTION = {
    "avg_loneliness": "lower",
    "avg_affordability_ratio": "lower",
    "avg_rent_per_person": "lower",
    "cumulative_public_cost": "lower",
    "total_public_cost": "lower",
    "low_income_access_rate": "higher",
    "number_people_housed": "higher",
    "share_matched": "higher",
    "match_rate": "higher",
    "avg_space_per_person": "higher",
}


def _rank_best_first(mean_by_scenario, outcome):
    # Rank scenarios 1..n with 1 = best, following the outcome's direction.
    ascending = OUTCOME_DIRECTION.get(outcome, "lower") == "lower"
    return mean_by_scenario.rank(ascending=ascending, method="min")


def sweep_scenario_ranking(param, values, outcomes, scenarios=None, steps=300,
                           seeds=range(5), window=50, base_kwargs=None, verbose=True):
    # For each value of `param`, run every scenario across seeds and record each
    # scenario's mean outcome and its rank. The swept value goes into base_kwargs,
    # so every scenario sees the same world and only the policy levers differ.
    # Returns a long DataFrame: [param, value, outcome, scenario, mean, std, rank].
    scenarios = scenarios or scenario_dict()
    base_kwargs = base_kwargs or {}
    records = []
    for v in values:
        if verbose:
            print(f"  {param} = {v} ...", flush=True)
        bk = {**base_kwargs, param: v}
        _, tidy = run_scenarios_multiseed(
            scenarios, steps=steps, seeds=seeds, window=window, base_kwargs=bk
        )
        means = tidy.groupby("scenario")[outcomes].mean()
        stds = tidy.groupby("scenario")[outcomes].std()
        for outcome in outcomes:
            ranks = _rank_best_first(means[outcome], outcome)
            for scen in means.index:
                records.append({
                    "param": param, "value": v, "outcome": outcome, "scenario": scen,
                    "mean": means.loc[scen, outcome], "std": stds.loc[scen, outcome],
                    "rank": int(ranks.loc[scen]),
                })
    return pd.DataFrame(records)


def ranking_stability(sweep_df, outcome):
    # Returns (rank_matrix, stability_table) for one outcome.
    # rank_matrix: rows = swept values, cols = scenarios, entries = rank (1 = best).
    # stability_table: per value, Spearman correlation of the ranking against the
    # first value's ranking, plus whichever scenario is currently ranked #1.
    sub = sweep_df[sweep_df["outcome"] == outcome]
    rank_mat = sub.pivot(index="value", columns="scenario", values="rank").sort_index()
    mean_mat = sub.pivot(index="value", columns="scenario", values="mean").sort_index()
    ref = rank_mat.iloc[0]
    higher_better = OUTCOME_DIRECTION.get(outcome, "lower") == "higher"
    rows = []
    for v in rank_mat.index:
        rows.append({
            "value": v,
            "spearman_vs_first": rank_mat.loc[v].corr(ref, method="spearman"),
            "top_scenario": (mean_mat.loc[v].idxmax() if higher_better
                             else mean_mat.loc[v].idxmin()),
        })
    return rank_mat, pd.DataFrame(rows).set_index("value")


def summarize_rank_stability(sweep_df):
    # One row per outcome: whether the #1 scenario stays constant across the sweep,
    # which scenarios ever held #1, and the worst rank correlation vs the first value
    # (1.0 = the order never changed).
    param = sweep_df["param"].iloc[0]
    rows = []
    for outcome in sweep_df["outcome"].unique():
        _, stab = ranking_stability(sweep_df, outcome)
        top_seen = sorted(stab["top_scenario"].unique())
        rows.append({
            "param": param,
            "outcome": outcome,
            "top_stable": len(top_seen) == 1,
            "top_scenarios_seen": ", ".join(top_seen),
            "min_spearman_vs_first": round(float(stab["spearman_vs_first"].min()), 3),
        })
    return pd.DataFrame(rows)


def plot_rank_bumps(sweep_df, outcome):
    # Bump chart: scenario rank (1 = best) as the swept parameter changes.
    # A line that changes height is a scenario whose ranking moved.
    sub = sweep_df[sweep_df["outcome"] == outcome]
    rank_mat = sub.pivot(index="value", columns="scenario", values="rank").sort_index()
    param = sub["param"].iloc[0]
    plt.figure(figsize=(10, 6))
    for scen in rank_mat.columns:
        plt.plot(rank_mat.index, rank_mat[scen], marker="o", label=scen)
    plt.gca().invert_yaxis()
    plt.yticks(range(1, rank_mat.shape[1] + 1))
    plt.xlabel(param)
    plt.ylabel("Rank (1 = best)")
    plt.title(f"Scenario ranking on {outcome} as {param} varies")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

### Setting up the sweeps

In [ ]:
# Which parameters to sweep and over what ranges. Every (value x scenario x seed)
# is one full simulation, so keep this trimmed to keep the runtime sane.
ROBUSTNESS_CONFIG = {
    "sweeps": {
        "n_students":             [200, 300, 400, 500, 600],   # demand pressure vs fixed stock
        "construction_lag_ticks": [6, 12, 24, 36, 48],         # do supply-side builds land in time?
        "reference_gap":          [100, 200, 300, 400],        # strength of price -> preference channel
        "interaction_quality_shared": [0.50, 0.60, 0.72, 0.85, 1.00],  # how much shared living eases loneliness
    },
    "key_outcomes": [
        "avg_loneliness", "avg_affordability_ratio",
        "low_income_access_rate", "cumulative_public_cost",
    ],
    # A readable subset of scenarios. Use list(scenario_dict()) for all nine.
    "scenarios": [
        "baseline_no_subsidy", "post_2026", "studio_reduction",
        "room_subsidy", "object_subsidy", "platform31_crossfinance",
        "full_intervention",
    ],
    "seeds": range(5),                 # raise for smoother ranks (and longer runtime)
    "steps": RUN_CONFIG["steps"],
    "window": RUN_CONFIG["window"],
}

_n_runs = (sum(len(v) for v in ROBUSTNESS_CONFIG["sweeps"].values())
           * len(ROBUSTNESS_CONFIG["scenarios"])
           * len(list(ROBUSTNESS_CONFIG["seeds"])))
print(f"This robustness analysis runs about {_n_runs} simulations "
      f"({ROBUSTNESS_CONFIG['steps']} steps each).")
print("Trim ROBUSTNESS_CONFIG['sweeps'] / 'scenarios' / 'seeds' if that's too slow.")

### Run the sweeps and read off the stability

In [ ]:
# Run every sweep. I keep the results per parameter so each one can be inspected.
scen_subset = {k: scenario_dict()[k] for k in ROBUSTNESS_CONFIG["scenarios"]}

robustness_results = {}
for _param, _values in ROBUSTNESS_CONFIG["sweeps"].items():
    print(f"Sweeping {_param} ...")
    robustness_results[_param] = sweep_scenario_ranking(
        param=_param,
        values=_values,
        outcomes=ROBUSTNESS_CONFIG["key_outcomes"],
        scenarios=scen_subset,
        steps=ROBUSTNESS_CONFIG["steps"],
        seeds=ROBUSTNESS_CONFIG["seeds"],
        window=ROBUSTNESS_CONFIG["window"],
        base_kwargs=MODEL_PARAMS,
    )

# Headline answer: does the ranking hold? One block of rows per swept parameter.
stability = pd.concat(
    [summarize_rank_stability(df) for df in robustness_results.values()],
    ignore_index=True,
)
print("\n=== RANK STABILITY SUMMARY ===")
print("top_stable             : did the #1 scenario stay the same across the sweep?")
print("min_spearman_vs_first  : worst-case rank correlation vs the first value")
print("                         (1.0 = order never changed; lower = it moved)\n")
stability

### Plotting the rank movement

In [ ]:
# Bump charts: scenario rank vs the swept parameter, for the headline outcome.
HEADLINE_OUTCOME = "avg_loneliness"
for _param, _df in robustness_results.items():
    plot_rank_bumps(_df, HEADLINE_OUTCOME)

# Zoom into one (parameter, outcome) pair, e.g. demand pressure on loneliness:
rank_mat, stab = ranking_stability(robustness_results["n_students"], "avg_loneliness")
print("Ranks (rows = n_students, cols = scenario, 1 = best):")
print(rank_mat)
print("\nStability vs first value:")
print(stab.round(3))

## 13. Global sensitivity analysis (Morris → Sobol)

The previous OFAT sweeps and bump charts are local sensitivity analyses. This section complements that with a global sensitivity analysis, exploring the full parameter space to identify which structural parameters drive model outcomes.

Using **SALib**, the analysis does two steps:

1. **Morris screening** identifies influential parameters (`μ*`) and potential interaction effects (`σ`). This is a gateway to Sobol analyses - full Sobol analysis across 39 parameters would be computationally too expensive.
2. **Sobol analysis** decomposes output variance into main effects (`S1`) and total effects (`ST`).

Interaction effect strength can be read from the difference between `S1` and `ST` (graph below)

Because the model is stochastic, results are averaged across multiple seeds. Policy settings are held fixed by default, but `contrast_scenario` can be used to analyse sensitivity of policy effects instead.

### Setup: parameters, bounds, evaluator

In [ ]:
# Global sensitivity analysis with SALib: Morris screens all structural parameters,
# then Sobol quantifies main + interaction effects on the selected top few.

from SALib.sample.morris import sample as morris_sample
from SALib.analyze.morris import analyze as morris_analyze
from SALib.analyze import sobol as sobol_analyzer
try:
    from SALib.sample import sobol as sobol_sampler          # SALib >= 1.4.5
except ImportError:                                          # older SALib name
    from SALib.sample import saltelli as sobol_sampler


# Integer-valued parameters: SALib samples continuously, so round these per draw.
INT_PARAMS = {
    "n_students", "initial_studios", "initial_shared",
    "construction_lag_ticks", "shared_household_size",
}

def _coerce(name, value):
    return int(round(value)) if name in INT_PARAMS else float(value)


# Candidate structural / behavioural parameters and the ranges to explore.
# (Policy levers are NOT here: policy is held fixed so we measure how STRUCTURE
# moves the outcome. Booleans / categoricals are excluded — Sobol needs numeric
# inputs.)
SA_BOUNDS = {
    "n_students":                 [200, 600],
    "initial_studios":            [40, 120],
    "initial_shared":             [80, 200],
    "move_probability":           [0.05, 0.6],
    "reference_gap":              [100, 400],
    "interaction_quality_studio": [0.10, 0.50],
    "interaction_quality_shared": [0.40, 1.00],
    "rent_adjustment_speed":      [0.005, 0.05],
    "vacancy_sensitivity":        [0.01, 0.08],
    "rent_inertia":               [0.05, 0.50],
    "studio_rent_eq":             [600, 900],
    "shared_rent_eq":             [400, 650],
    "operating_cost":             [350, 650],
    "coverage_ratio":             [0.70, 1.00],
    "conversion_rate":            [0.02, 0.25],
    "investor_withdrawal_rate":   [0.01, 0.15],
    "studio_build_cost":          [350, 650],
    "shared_build_cost":          [250, 500],
    "gamma_studio":               [0.01, 0.06],
    "gamma_shared":               [0.01, 0.06],
    "kappa_buffer_pct":           [0.05, 0.40],
    "construction_lag_ticks":     [6, 48],
    "shared_household_size":      [2, 8],
}

def make_problem(names):
    # SALib problem dict for a chosen subset of parameters.
    return {"num_vars": len(names), "names": list(names),
            "bounds": [SA_BOUNDS[n] for n in names]}

SA_PROBLEM_FULL = make_problem(list(SA_BOUNDS))


SA_CONFIG = {
    "scenario": "full_intervention",   # policy held fixed while structure varies
    "outcome": "avg_loneliness",
    # Set contrast_scenario to compare two policies, e.g. "baseline_no_subsidy":
    # the output becomes outcome(scenario) - outcome(contrast), i.e. the POLICY GAP,
    # so Sobol then tells you which structural knobs most destabilise the ranking.
    "contrast_scenario": None,
    "steps": 200,                      # shorter than the main run to save time
    "seeds": range(3),                 # averaged per evaluation to tame seed noise
    "window": RUN_CONFIG["window"],
    "morris_trajectories": 10,         # Morris evals = trajectories * (num_params + 1)
    "morris_levels": 4,
    "screen_keep": 6,                  # how many top params (by mu_star) go to Sobol
    "sobol_N": 64,                     # Sobol evals = N * (2k+2) or N * (k+2); use >=256 for real runs
    "second_order": False,             # True also estimates pairwise interactions (costlier)
}


def _sa_mean_outcome(base_kw, scenario_params, outcome, steps, seeds, window):
    # Average of `outcome` in the time period selected, averaged across seeds (one number per eval).
    vals = []
    for s in seeds:
        _, mdf, _ = run_model(steps=steps, **{**base_kw, **scenario_params, "seed": int(s)})
        tail = mdf.tail(min(window, len(mdf)))
        vals.append(float(tail[outcome].mean()))
    return float(np.mean(vals))


def make_evaluator(names, base_kwargs, scenario_params, outcome,
                   steps, seeds, window, contrast_params=None):
    # Returns f(x_row) -> single number for SALib. Sampled params override the
    # baseline, everything else stays at MODEL_PARAMS. If contrast_params is given,
    # the output is the difference between the two scenarios (the policy gap).
    def evaluate(x_row):
        kw = dict(base_kwargs)
        for name, val in zip(names, x_row):
            kw[name] = _coerce(name, val)
        y = _sa_mean_outcome(kw, scenario_params, outcome, steps, seeds, window)
        if contrast_params is not None:
            y -= _sa_mean_outcome(kw, contrast_params, outcome, steps, seeds, window)
        return y
    return evaluate


def run_model_over_samples(X, evaluate, label=""):
    # Evaluate every SALib sample row, printing progress now and then.
    n = X.shape[0]
    Y = np.empty(n)
    for i in range(n):
        Y[i] = evaluate(X[i])
        if i == 0 or (i + 1) % 25 == 0 or i == n - 1:
            print(f"  {label} eval {i + 1}/{n}", flush=True)
    return Y


# Runtime estimate (model evaluations, each = len(seeds) simulations)
_D = SA_PROBLEM_FULL["num_vars"]
_k = SA_CONFIG["screen_keep"]
_per_eval = len(list(SA_CONFIG["seeds"])) * (2 if SA_CONFIG["contrast_scenario"] else 1)
_morris_evals = SA_CONFIG["morris_trajectories"] * (_D + 1)
_sobol_evals = SA_CONFIG["sobol_N"] * ((2 * _k + 2) if SA_CONFIG["second_order"] else (_k + 2))
print(f"Morris: ~{_morris_evals} evaluations  ({_morris_evals * _per_eval} simulations)")
print(f"Sobol : ~{_sobol_evals} evaluations  ({_sobol_evals * _per_eval} simulations)")
print("Lower morris_trajectories / sobol_N / steps / seeds to speed up; raise for fidelity.")


### Stage 1 – Morris screening

In [ ]:
# Morris screening.
_scenario_params = scenario_dict()[SA_CONFIG["scenario"]]
_contrast_params = (scenario_dict()[SA_CONFIG["contrast_scenario"]]
                    if SA_CONFIG["contrast_scenario"] else None)

evaluate_full = make_evaluator(
    SA_PROBLEM_FULL["names"], MODEL_PARAMS, _scenario_params, SA_CONFIG["outcome"],
    SA_CONFIG["steps"], SA_CONFIG["seeds"], SA_CONFIG["window"], _contrast_params,
)

X_morris = morris_sample(
    SA_PROBLEM_FULL, N=SA_CONFIG["morris_trajectories"],
    num_levels=SA_CONFIG["morris_levels"],
)
Y_morris = run_model_over_samples(X_morris, evaluate_full, label="morris")

Mi = morris_analyze(
    SA_PROBLEM_FULL, X_morris, Y_morris,
    num_levels=SA_CONFIG["morris_levels"], print_to_console=False,
)

# mu_star = overall influence; sigma = how much the effect depends on other params.
morris_df = (pd.DataFrame({
    "param": SA_PROBLEM_FULL["names"],
    "mu_star": Mi["mu_star"],
    "sigma": Mi["sigma"],
}).sort_values("mu_star", ascending=False).reset_index(drop=True))

survivors = morris_df.head(SA_CONFIG["screen_keep"])["param"].tolist()
print("Top parameters by mu_star (these go on to Sobol):", survivors)
morris_df

In [ ]:
# Morris plots: the influence ranking, and the mu_star vs sigma map.
plt.figure(figsize=(9, 5))
plt.barh(morris_df["param"][::-1], morris_df["mu_star"][::-1])
plt.xlabel("mu* (overall influence)")
plt.title(f"Morris screening on {SA_CONFIG['outcome']}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(morris_df["mu_star"], morris_df["sigma"])
for _, r in morris_df.iterrows():
    plt.annotate(r["param"], (r["mu_star"], r["sigma"]),
                 fontsize=8, xytext=(4, 2), textcoords="offset points")
plt.xlabel("mu* (influence)")
plt.ylabel("sigma (interactions / nonlinearity)")
plt.title("Morris: high sigma = effect depends on the other parameters")
plt.tight_layout()
plt.show()

### Stage 2 – Sobol on the top parameters
`screen_keep` parameters from the top are kept for this stage

In [ ]:
# Sobol on the survivors.
problem_sobol = make_problem(survivors)

X_sobol = sobol_sampler.sample(
    problem_sobol, SA_CONFIG["sobol_N"], calc_second_order=SA_CONFIG["second_order"],
)
# Same evaluator logic, now only over the survivor parameters.
evaluate_sobol = make_evaluator(
    problem_sobol["names"], MODEL_PARAMS, _scenario_params, SA_CONFIG["outcome"],
    SA_CONFIG["steps"], SA_CONFIG["seeds"], SA_CONFIG["window"], _contrast_params,
)
Y_sobol = run_model_over_samples(X_sobol, evaluate_sobol, label="sobol")

Si = sobol_analyzer.analyze(
    problem_sobol, Y_sobol,
    calc_second_order=SA_CONFIG["second_order"], print_to_console=False,
)

# S1 = main effect (this param alone); ST = total effect (incl. interactions).
# A big ST-S1 gap means the parameter acts mostly THROUGH interactions.
sobol_df = (pd.DataFrame({
    "param": survivors,
    "S1": Si["S1"], "S1_conf": Si["S1_conf"],
    "ST": Si["ST"], "ST_conf": Si["ST_conf"],
}).assign(interaction=lambda d: d["ST"] - d["S1"])
  .sort_values("ST", ascending=False).reset_index(drop=True))
print("S1 = main effect, ST = total effect, interaction = ST - S1")
sobol_df.round(4)

In [ ]:
# Sobol plot: first-order vs total-order indices with confidence bars.
import numpy as _np
order = sobol_df["param"].tolist()
x = _np.arange(len(order))
w = 0.38
plt.figure(figsize=(9, 5))
plt.bar(x - w / 2, sobol_df["S1"], w, yerr=sobol_df["S1_conf"], capsize=3, label="S1 (main)")
plt.bar(x + w / 2, sobol_df["ST"], w, yerr=sobol_df["ST_conf"], capsize=3, label="ST (total)")
plt.xticks(x, order, rotation=30, ha="right")
plt.ylabel("Sobol index")
plt.title(f"Sobol indices on {SA_CONFIG['outcome']}")
plt.legend()
plt.tight_layout()
plt.show()

# Optional pairwise-interaction heatmap (only if second_order was estimated).
if SA_CONFIG["second_order"]:
    S2 = _np.array(Si["S2"])
    plt.figure(figsize=(6, 5))
    plt.imshow(S2, cmap="viridis")
    plt.colorbar(label="S2 (pairwise interaction)")
    plt.xticks(range(len(survivors)), survivors, rotation=90)
    plt.yticks(range(len(survivors)), survivors)
    plt.title("Second-order Sobol indices")
    plt.tight_layout()
    plt.show()

## 14. Interpretation

To fill in based on previous section. Extended reporting in the assignment submission (link)

## 15. Potential extentions

- Estimate the behavioural parameters from survey data instead of hand-tuning them.
- Make the developers' expectations adaptive instead of rule-based.
- Add variation in shared-household composition and turnover